🔗 [Back to Table of Contents](https://github.com/najaeda/najaeda-tutorials#-table-of-contents)

# Chapter 3: Editing a Netlist

In this chapter, we'll revisit the small full adder design from Chapter 1 and explore how to modify it using `najaeda`'s editing API.

As a reminder, here's the schematic of the full adder:

![FullAdder](https://raw.githubusercontent.com/najaeda/najaeda-tutorials/main/images/fulladder.png)

Let’s begin by setting up the environment — we’ll install **najaeda** and write the design to a local file so we can edit it.

In [ ]:
!pip install najaeda

In [ ]:
%%writefile fulladder.v
module halfadder(
    input a,
    input b,
    output sum,
    output carry
);
    and carry_and(carry, a, b);
    xor sum_xor(sum, a, b);
endmodule

module fulladder(
    input a,
    input b,
    input cin,
    output sum,
    output cout
);
    wire sum1, carry1, carry2;
    halfadder ha1(
        .a(a),
        .b(b),
        .sum(sum1),
        .carry(carry1)
    );
    halfadder ha2(
        .a(sum1),
        .b(cin),
        .sum(sum),
        .carry(carry2)
    );
    or cout_or(cout, carry1, carry2);
endmodule

We’ll begin by importing the `najaeda` library and loading the full adder netlist for editing.

Let's also dump a dot diagram of the netlist for reference.

In [ ]:

from najaeda import netlist

def display_dot(instance):
    instance.dump_full_dot(f"{instance.get_name()}.dot")
    !dot -Tpng {instance.get_name()}.dot -o {instance.get_name()}.png
    from IPython.display import Image, display
    display(Image(filename=f"{instance.get_name()}.png"))

top = netlist.load_verilog('fulladder.v')
print(f"Design name: {top.get_name()} loaded")
display_dot(top)

## Editing the netlist

### Renaming

Let's start by renaming objects in the netlist.

In [ ]:
ha1 = top.get_child_instance('ha1')
print(f"Renaming instance name: {ha1.get_name()} with model: {ha1.get_model_name()} to 'halfadder1'")
ha1.set_name('halfadder1')
print(f"After renaming, renamed instance name: {ha1.get_name()} with model: {ha1.get_model_name()}")

As the instance `ha1` (now `halfadder1`) is under the top, there is no need to uniquify anything in the netlist to perform safely this renaming.

Let's rename back `halfadder1` to previous name `ha1` to come back to initial situation.

In [ ]:
ha1.set_name('ha1')
print(f"instance name: {ha1.get_name()} with model: {ha1.get_model_name()}")


Let's rename now a lower instance: the `and` `carry_and` gate under `ha1`.

In [ ]:
ha1_and = ha1.get_child_instance('carry_and')
print(f"Instance name: {ha1_and.get_name()} with model: {ha1_and.get_model_name()}")
ha1_and.set_name('and1')
print(f"After renaming, renamed instance name: {ha1_and.get_name()} with model: {ha1_and.get_model_name()}")
print(f"After renaming, {ha1.get_name()} has model {ha1.get_model_name()}")


As we can see, after renaming `ha1/carry_and` to `and1`, `and1` model is untouched, but `ha1` has been uniquified. This is expected and required as `ha1` and `ha2` share same model `halfadder` and the only the renaming of `ha1/carry_and` was requested.

Therefore, a unique model for `ha1`: a clone of `halfadder` has been created.

Let's use `dot` dumping to visualize the netlist after the renaming.


In [ ]:
display_dot(top)

In [ ]:
sum1_net = top.get_net('sum1')
print(f"Net name: {sum1_net.get_name()}")
sum1_net.set_name('sum1_renamed')
print(f"After renaming, net name: {sum1_net.get_name()}")
display_dot(top)

### Editing the connectivity